In [1]:
# setup
try:
  import google.colab
  IN_COLAB = True
  print("Running as a Colab notebook")
  %pip install transformer_lens
  %pip install circuitsvis
except:
  IN_COLAB = False
  print("Running as a Jupyter notebook - intended for development only!")

Running as a Jupyter notebook - intended for development only!


In [2]:
import einops
from fancy_einsum import einsum
from dataclasses import dataclass
# from transformer_lens import HookedTransformer
from transformer_lens import TransformerBridge
import torch as t
import torch.nn as nn
import numpy as np
import math
from transformer_lens.utilities import gelu_new, tokenize_and_concatenate
import tqdm.auto as tqdm

/Users/dhirana/Codes/paper-replications/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1031 18:21:15.059000 7855 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
reference_gpt2 = TransformerBridge.boot_transformers("gpt2")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10420.11it/s]


# Understanding Transformer Inputs + Outputs

Transformers exist to model text. So for eg. a gpt-2 style transformer, we feed in language, and the model generates a probability distribution over tokens. To be more technical, what it does is:

1. we need in a seq of length N
2. it samples from the probability distribution over the N+1-th word
3. uses that to construct a new seq of length N+1
4. feeds that new seq into the model
5. to get a probability distribution over the N+2-th word and so on.

Core point: we make the transformer have causal attention. It can only move information forwards in the sequence. Transformer is autoregressive cos it only predicts future words based on past data. 

To feed a word like "cat" into a model, we must turn it into a list of numbers (aka a vector).

This is a 2-step process: Token IDs(Integers) and Embeddings (Vectors)
- Step 1: give every word an ID number (integer)
- Step 2: use that ID to look up a list of numbers (vectors). Cos for eg if in step 1, we decided "cat" is 2. AN integer like 2 here is just a label without meaning. To let the model actl calcualte, we use a **lookup table** (Embedding Matrix). Sort of a grid where every row corresponds to a word's ID and contains a list of coord (a vector). Now cat is represented by that vector instead. 


But why not just use integers?
- cos integers force relationships and false hierarchy. eg if cat is 1, dog is 2 and airplane is 3, the model might think cat + dog = airplane. which is dumb
- vector also act as set of slider coordinates for diff traits


Btw the more powerful a model, the more columns its vectors has. 

### But how to slice text into those ID numbers (integers) in the first place?

Via the Tokenisation Process
- to build this lookup table, we need a fixed vocabulary
    - idea 1: whole words (dictionary) ie. give every singel word in the English language its own ID number. This breaks however if someone types a typo (catt) or random URL, or a new slang...model hits an 'out of vocabulary error' cos it doesn't have an ID for it
    - idea 2: raw characters (256 ASCII characters) ie. just giving every letter/number/punctiuation its own ID. BUT, this loses the structure of language making sequences massive. Forces model to waste brainpower learning how to spell words instead of understanding what they mean.
- SOLUTION: Byte-Pair Encoding(BPE)
    - Sub-word tokenisation ie. starting with single characters so they can spell anything. And then selectively gluing the most common pairs tgt to create whole/partial words. 
    - 1. Basics: start w tiny vocab of just 256 raw characters
    - 2. Find common pairs: ie. looks at millions of pages of text and asks which 2 tokens appear next to each other the most often
    - 3. merge: glues those 2 togterhter and gives that new combo its own brand-new unique ID
    - 4. repeat: until it has a huge vocab of ard 32k-50k tokens

In [ ]:
sorted_vocab = sorted(list(reference_gpt2.tokenizer.vocab.items()), key=lambda n:n[1])

print(sorted_vocab[:20])
print()
print(sorted_vocab[250:270])
print()
print(sorted_vocab[990:1010])
print()

#. Note - you might see the character Ġ in front of some tokens. This is a special token that indicates that the token begins with a space. Tokens with a leading space vs not are different.

[('!', 0), ('"', 1), ('#', 2), ('$', 3), ('%', 4), ('&', 5), ("'", 6), ('(', 7), (')', 8), ('*', 9), ('+', 10), (',', 11), ('-', 12), ('.', 13), ('/', 14), ('0', 15), ('1', 16), ('2', 17), ('3', 18), ('4', 19)]

[('ľ', 250), ('Ŀ', 251), ('ŀ', 252), ('Ł', 253), ('ł', 254), ('Ń', 255), ('Ġt', 256), ('Ġa', 257), ('he', 258), ('in', 259), ('re', 260), ('on', 261), ('Ġthe', 262), ('er', 263), ('Ġs', 264), ('at', 265), ('Ġw', 266), ('Ġo', 267), ('en', 268), ('Ġc', 269)]

[('Ġprodu', 990), ('Ġstill', 991), ('led', 992), ('ah', 993), ('Ġhere', 994), ('Ġworld', 995), ('Ġthough', 996), ('Ġnum', 997), ('arch', 998), ('imes', 999), ('ale', 1000), ('ĠSe', 1001), ('ĠIf', 1002), ('//', 1003), ('ĠLe', 1004), ('Ġret', 1005), ('Ġref', 1006), ('Ġtrans', 1007), ('ner', 1008), ('ution', 1009)]



In [ ]:
sorted_vocab[-20:]
# shows the tail end of gpt-2's vocabulary list. specifically the final 20 tokens out of its total 50,257 vocabulary size. 

# Ġ character: is a placeholder for a space. cos handling raw space characters can get messy + buggy
# Consider ('ominated', 50251) and ('Compar', 50249) - those are sub-words. Byte-pair encoding means the tokeniser doesn't always store whole words. Eg. 'ominated' can be reused to build dominated, nominated, culminated etc.
# very last item in the list is ('<|endoftext|>', 50256) --> special control token called a End-of-Sequence token (EOS). It doesn't actl rep a real world we wld type. Instead it acts like a structural bookmark. When the model is reading/writing text, this token tells the model to stop generating text at X position. 

[('Revolution', 50237),
 ('Ġsnipers', 50238),
 ('Ġreverted', 50239),
 ('Ġconglomerate', 50240),
 ('Terry', 50241),
 ('794', 50242),
 ('Ġharsher', 50243),
 ('Ġdesolate', 50244),
 ('ĠHitman', 50245),
 ('Commission', 50246),
 ('Ġ(/', 50247),
 ('âĢ¦."', 50248),
 ('Compar', 50249),
 ('Ġamplification', 50250),
 ('ominated', 50251),
 ('Ġregress', 50252),
 ('ĠCollider', 50253),
 ('Ġinformants', 50254),
 ('Ġgazed', 50255),
 ('<|endoftext|>', 50256)]

In [6]:
print(reference_gpt2.to_str_tokens("Ralph"))
print(reference_gpt2.to_str_tokens(" Ralph"))
print(reference_gpt2.to_str_tokens(" ralph"))
print(reference_gpt2.to_str_tokens("ralph"))

['<|endoftext|>', 'R', 'alph']
['<|endoftext|>', ' Ralph']
['<|endoftext|>', ' r', 'alph']
['<|endoftext|>', 'ral', 'ph']


In [7]:
reference_gpt2.to_str_tokens("56873+3184623=123456789-1000000000")

['<|endoftext|>',
 '568',
 '73',
 '+',
 '318',
 '46',
 '23',
 '=',
 '123',
 '45',
 '67',
 '89',
 '-',
 '1',
 '000000',
 '000']

Key Takeaways:
* We learn a dictionary of vocab of tokens (sub-words).
* We (approx) losslessly convert language to integers via tokenizing it.
* We convert integers to vectors via a lookup table.
* Note: input to the transformer is a sequence of *tokens* (ie integers), not vectors